# (b) Train YOLO-seg ngập đường phố từ dataset của `auto_label.ipynb`

Chạy trên Kaggle, **GPU bật** (P100 hoặc T4), **Internet bật** (để tải `ultralytics` và weight pretrained).

**Input**: output của notebook auto-label. Bấm **Add Input → Your Work → Notebooks**, chọn notebook đã chạy `auto_label.ipynb` (vd. `notebook7d0b05de00`). Kaggle gắn **version output mới nhất** của notebook đó; chạy lại auto-label thì vào Input bấm cập nhật version. Có thể dùng Kaggle Dataset tạo từ folder `dataset/` thay cho notebook output.

Notebook tự tìm `dataset/data.yaml` trong `/kaggle/input` và **dừng nếu dataset có `NOT_READY_FOR_TRAINING.txt`**, trừ khi đặt `ALLOW_NOT_READY=True`.

Class: `0 flood`, `1 road`, `2–6 car_L0–car_L4`. Test chỉ có nhãn **flood thật**, nên chỉ đọc dòng `flood` khi đánh giá trên test.

Output trong `/kaggle/working`: `runs/seg/...` (log, biểu đồ, `weights/best.pt`), `best_flood_seg.pt`, `predict_samples/`.

In [ ]:
# ======================= CẤU HÌNH TRAIN =======================
MODEL='yolo11s-seg.pt'     # n (nhanh) / s (cân bằng, mặc định) / m (chính xác hơn, chậm hơn ~2.5x). Dataset ~1000 ảnh: s hoặc m.
EPOCHS=100
PATIENCE=30                # dừng sớm khi val không cải thiện sau 30 epoch
IMGSZ=640
BATCH=16                   # hết VRAM thì giảm 8; đặt -1 để Ultralytics tự chọn
WORKERS=2                  # Kaggle có 2-4 CPU
SEED=42
ALLOW_NOT_READY=False      # True: vẫn train khi auto-label ghi NOT_READY_FOR_TRAINING.txt (chỉ để thử nghiệm)
RESUME=False               # True: tiếp tục từ runs/seg/weights/last.pt của phiên này, hoặc của version trước được Add Input
N_PREDICT_SAMPLES=24       # số ảnh val/test vẽ dự đoán để xem bằng mắt

In [ ]:
%pip -q install ultralytics

In [ ]:
# ---------- Tìm dataset, kiểm tra cổng chất lượng, ghi data.yaml với đường dẫn thật ----------
from pathlib import Path
import json, yaml, collections, shutil
INPUT_ROOT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
cands=[p.parent for p in INPUT_ROOT.rglob('data.yaml') if (p.parent/'images'/'train').is_dir() and (p.parent/'labels'/'train').is_dir()]
if not cands:
    raise FileNotFoundError('Không tìm thấy dataset/data.yaml (kèm images/train, labels/train) trong /kaggle/input. '
                            'Add Input -> Your Work -> Notebooks -> notebook auto-label (version đã chạy xong).')
# Nhiều ứng viên (vd. nhiều version được add): ưu tiên bản có READY_FOR_TRAINING.txt, rồi bản mới nhất.
cands.sort(key=lambda d:((d/'READY_FOR_TRAINING.txt').exists(),(d/'data.yaml').stat().st_mtime),reverse=True)
DATA_DIR=cands[0]
print('Dataset:',DATA_DIR, *(['(các ứng viên khác: '+', '.join(map(str,cands[1:]))+')'] if len(cands)>1 else []))

if (DATA_DIR/'NOT_READY_FOR_TRAINING.txt').exists():
    print((DATA_DIR/'NOT_READY_FOR_TRAINING.txt').read_text())
    if not ALLOW_NOT_READY:
        raise RuntimeError('Dataset CHƯA sẵn sàng (xem lý do ở trên). Sửa ở auto_label.ipynb rồi chạy lại, '
                           'hoặc đặt ALLOW_NOT_READY=True nếu chỉ muốn train thử.')
    print('[WARNING] ALLOW_NOT_READY=True: train trên dataset chưa đạt cổng chất lượng.')
elif (DATA_DIR/'READY_FOR_TRAINING.txt').exists():
    print((DATA_DIR/'READY_FOR_TRAINING.txt').read_text())
else:
    print('[WARNING] Không có READY/NOT_READY_FOR_TRAINING.txt: dataset không phải từ auto_label.ipynb bản mới.')

cfg=yaml.safe_load((DATA_DIR/'data.yaml').read_text())
NAMES=cfg['names']; NAMES={int(k):v for k,v in (NAMES.items() if isinstance(NAMES,dict) else enumerate(NAMES))}
# data.yaml gốc trỏ tới /kaggle/working/dataset của notebook auto-label; ở đây dataset nằm trong /kaggle/input (chỉ đọc).
has_test=any((DATA_DIR/'images'/'test').glob('*'))
cfg.update({'path':str(DATA_DIR),'train':'images/train','val':'images/val','names':NAMES})
if has_test: cfg['test']='images/test'
else: cfg.pop('test',None)
DATA_YAML=WORK/'data_train.yaml'
DATA_YAML.write_text(yaml.safe_dump(cfg,sort_keys=False,allow_unicode=True))
print('data.yaml dùng để train:',DATA_YAML); print(DATA_YAML.read_text())

# Thống kê nhanh: số ảnh, ảnh có từng class, số instance theo split.
for split in ('train','val','test'):
    lbls=sorted((DATA_DIR/'labels'/split).glob('*.txt'))
    if not lbls: continue
    img_c=collections.Counter(); inst_c=collections.Counter(); empty=0
    for f in lbls:
        cls=[int(l.split()[0]) for l in f.read_text().splitlines() if l.strip()]
        empty+=not cls; inst_c.update(cls); img_c.update(set(cls))
    print(f'{split}: {len(lbls)} ảnh ({empty} nhãn rỗng) |',
          ', '.join(f'{NAMES[c]}: {img_c[c]} ảnh/{inst_c[c]} inst' for c in sorted(NAMES)))

In [ ]:
# ---------- Train ----------
from ultralytics import YOLO
import torch
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU - bật Accelerator GPU trong Settings')
PROJECT=WORK/'runs'; RUN_NAME='seg'
lasts=[PROJECT/RUN_NAME/'weights'/'last.pt']+sorted(INPUT_ROOT.rglob('runs/seg/weights/last.pt'))
last=next((p for p in lasts if p.exists()),None)
if RESUME and last is None: raise FileNotFoundError('RESUME=True nhưng không thấy runs/seg/weights/last.pt (Add Input output của version train trước).')
if RESUME:
    print('Resume từ',last); model=YOLO(str(last)); results=model.train(resume=True)
else:
    model=YOLO(MODEL)
    results=model.train(data=str(DATA_YAML),epochs=EPOCHS,patience=PATIENCE,imgsz=IMGSZ,batch=BATCH,workers=WORKERS,seed=SEED,
                        device=0 if torch.cuda.is_available() else 'cpu',project=str(PROJECT),name=RUN_NAME,exist_ok=True,
                        plots=True,
                        flipud=0.0,fliplr=0.5)   # camera đường phố: lật ngang được, không lật dọc (trời xuống dưới)
SAVE_DIR=Path(model.trainer.save_dir); BEST=SAVE_DIR/'weights'/'best.pt'
shutil.copy(BEST,WORK/'best_flood_seg.pt')
print('Best weights:',BEST,'-> copy',WORK/'best_flood_seg.pt')

In [ ]:
# ---------- Đánh giá ----------
best=YOLO(str(BEST))
def report(split):
    m=best.val(data=str(DATA_YAML),split=split,imgsz=IMGSZ,batch=BATCH,device=0 if torch.cuda.is_available() else 'cpu',
               project=str(PROJECT),name=f'val_{split}',exist_ok=True,plots=True)
    print(f'\n== {split}: mask mAP50 = {m.seg.map50:.3f}, mask mAP50-95 = {m.seg.map:.3f} (trung bình các class có nhãn trong {split})')
    for i,c in enumerate(m.seg.ap_class_index):
        p,r,ap50,ap=m.seg.class_result(i)
        print(f'  {NAMES[int(c)]:8s} P={p:.3f} R={r:.3f} mask mAP50={ap50:.3f} mAP50-95={ap:.3f}')
    return m
val_metrics=report('val')
if has_test:
    # Test chỉ có nhãn flood thật (mask GT); road/car không được gán nhãn nên không có dòng cho chúng.
    test_metrics=report('test')

In [ ]:
# ---------- Vẽ dự đoán lên một số ảnh val/test để kiểm tra bằng mắt ----------
import random
imgs=[p for s in ('test','val') for p in sorted((DATA_DIR/'images'/s).glob('*'))]
random.Random(SEED).shuffle(imgs)
out=WORK/'predict_samples'
if out.exists(): shutil.rmtree(out)
best.predict([str(p) for p in imgs[:N_PREDICT_SAMPLES]],imgsz=IMGSZ,conf=0.25,save=True,project=str(WORK),name='predict_samples',exist_ok=True,verbose=False)
print('Ảnh dự đoán:',out)
print('Tải về: tab Output -> best_flood_seg.pt (dùng cho suy luận: YOLO(\'best_flood_seg.pt\').predict(...))')